# Smart MCQ Solver Challenge Inference Notebook
**Name:** Shobhit Raj  
**Roll No:** 24f2008744  
**Task:** Generating the final submission using DeBERTa-v3-large and Top-3 Logit Extraction.

In [3]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# 1. Tell W&B exactly which project to use (NO MORE DEFAULT NAME!)
# Change "your-project-name" to whatever you actually named your project in W&B
os.environ["WANDB_PROJECT"] = "24f2008744-t22026" 

# 2. Pull the secret key from Kaggle's vault
user_secrets = UserSecretsClient()
my_secret_key = user_secrets.get_secret("WANDB_API_KEY")

# 3. Log in to Weights & Biases
wandb.login(key=my_secret_key)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [4]:
# ==========================================
# INSTALL DEPENDENCIES (if needed)
# ==========================================
# !pip install datasets accelerate

import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from typing import Optional, Union

# ==========================================
# 1. CONFIGURATION (THE HEAVY ARTILLERY)
# ==========================================
# Large model for massive reasoning gains
MODEL_NAME = "microsoft/deberta-v3-large" 


KAGGLE_INPUT_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
TRAIN_DATA_PATH = f"{KAGGLE_INPUT_DIR}/train.csv"
TEST_DATA_PATH = f"{KAGGLE_INPUT_DIR}/test.csv"
OUTPUT_DIR = "./deberta-large-mcq-model"
SUBMISSION_PATH = "submission.csv"

INDEX_TO_LETTER = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# ==========================================
# 2. PREPARE TRAINING DATA
# ==========================================
print("--- STEP 1: Loading and Preprocessing Training Data ---")
train_df = pd.read_csv(TRAIN_DATA_PATH).fillna("")

label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df['label'] = train_df['answer'].map(label_map)

dataset = Dataset.from_pandas(train_df)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess_function(examples):
    prompts = [[context] * 5 for context in examples["prompt"]]
    choices = [
        [examples["A"][i], examples["B"][i], examples["C"][i], examples["D"][i], examples["E"][i]] 
        for i in range(len(examples["prompt"]))
    ]
    
    prompts_flat = sum(prompts, [])
    choices_flat = sum(choices, [])
    
    tokenized_examples = tokenizer(
        prompts_flat,
        choices_flat,
        truncation=True,
        max_length=256,
        padding=False
    )
    
    results = {
        k: [v[i : i + 5] for i in range(0, len(v), 5)] 
        for k, v in tokenized_examples.items()
    }
    results["labels"] = examples["label"]
    return results

print("Tokenizing training data...")
tokenized_dataset = dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=dataset.column_names
)

# ==========================================
# 3. CUSTOM DATA COLLATOR
# ==========================================
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None

    def __call__(self, features):
        labels = [feature.pop("labels") if "labels" in feature else feature.pop("label") for feature in features]
        
        for feature in features:
            for k in list(feature.keys()):
                if k not in ["input_ids", "attention_mask", "token_type_ids"]:
                    feature.pop(k, None)

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt",
        )
        
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

# ==========================================
# 4. FINE-TUNING
# ==========================================
print(f"\n--- STEP 2: Initializing {MODEL_NAME} ---")
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="no",
    save_strategy="epoch",
    # LOWER learning rate for the large model so it doesn't destabilize
    learning_rate=1e-5, 
    # BATCH SIZE 1: Crucial so the Kaggle T4 GPU doesn't run out of memory!
    per_device_train_batch_size=1, 
    # GRADIENT ACCUMULATION 8: Acts like a batch size of 8 without crashing memory
    gradient_accumulation_steps=8, 
    num_train_epochs=3, 
    weight_decay=0.01,
    fp16=False, # Keeping this False to avoid the gradient unscaling error
    report_to="wandb"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer)
)

print("Training massive model in progress... (This will take significantly longer, be patient!)")
trainer.train()

print(f"Training complete! Saving model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ==========================================
# 5. INFERENCE & SUBMISSION
# ==========================================
print("\n--- STEP 3: Generating Predictions on Test Data ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval() 

test_df = pd.read_csv(TEST_DATA_PATH).fillna("")
predictions = []

for index, row in test_df.iterrows():
    prompt = str(row['prompt'])
    choices = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    
    prompts = [prompt] * len(choices)
    
    inputs = tokenizer(
        prompts, 
        choices, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=256
    ).to(device)
    
    inputs = {k: v.unsqueeze(0) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0].cpu().numpy()
    
    top_3_indices = np.argsort(logits)[::-1][:3]
    top_3_letters = [INDEX_TO_LETTER[idx] for idx in top_3_indices]
    
    prediction_string = " ".join(top_3_letters)
    predictions.append(prediction_string)
    
    if (index + 1) % 50 == 0:
        print(f"Scored {index + 1}/{len(test_df)} test questions...")

print("\n--- STEP 4: Saving Final Submission ---")
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': predictions
})

submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Success! Your file '{SUBMISSION_PATH}' is ready to be submitted to the leaderboard.")

--- STEP 1: Loading and Preprocessing Training Data ---
Tokenizing training data...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


--- STEP 2: Initializing microsoft/deberta-v3-large ---


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias               

Training massive model in progress... (This will take significantly longer, be patient!)


Exception ignored in: <function WeakSet.__init__.<locals>._remove at 0x78112a17ec00>
Traceback (most recent call last):
  File "/usr/lib/python3.12/_weakrefset.py", line 39, in _remove
    def _remove(item, selfref=ref(self)):

KeyboardInterrupt: 
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


KeyboardInterrupt: 